# Merge and Preprocessing Code for Solar Weather and PV AC Power Output Dataset

This notebook documents the data-preparation pipeline for the processed dataset `Solar_Weather_Data_Processed.csv`.

**Important:** the two source inputs come from different Kaggle sources. The final dataset should be described as a curated processed benchmark dataset, not as a single raw field-measured dataset from one physical site unless the source documentation independently confirms that claim.

In [ ]:
# =========================
# 0. Imports and settings
# =========================

import os
import glob
import numpy as np
import pandas as pd

PV_FILE = "Plant_1_Generation_Data.csv"
WEATHER_FILE = "SolarPrediction.csv"
OUTPUT_FILE = "Solar_Weather_Data_Processed_reproduced.csv"

# Date window used by the published processed dataset.
START_TIMESTAMP = "2016-09-01 10:00:00"
END_TIMESTAMP = "2016-12-26 02:00:00"

# Location assumption used for solar-position features.
# Update this if you confirm a different location from the source documentation.
LATITUDE_DEG = 28.6
LONGITUDE_DEG = 77.2

RANDOM_SEED = 42

In [ ]:
# =========================
# 1. Helper functions
# =========================

def find_file(candidates, search_dirs=(".", "/content", "/mnt/data", "/kaggle/input", "/kaggle/working")):
    for d in search_dirs:
        for c in candidates:
            p = os.path.join(d, c)
            if os.path.isfile(p):
                return p
    for d in search_dirs:
        for c in candidates:
            hits = sorted(glob.glob(os.path.join(d, "**", c), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"Could not find any of these files: {candidates}")


def build_timestamp_from_weather(df):
    cols_lower = {c.lower(): c for c in df.columns}
    if "unixtime" in cols_lower:
        c = cols_lower["unixtime"]
        ts = pd.to_datetime(df[c], unit="s", errors="coerce")
        if ts.notna().mean() > 0.90:
            return ts

    date_candidates = [c for c in df.columns if c.lower() in ["date", "data", "formatted date", "formatted_date"]]
    time_candidates = [c for c in df.columns if c.lower() in ["time", "hour", "hh:mm", "time_h"]]
    if date_candidates and time_candidates:
        combined = df[date_candidates[0]].astype(str).str.strip() + " " + df[time_candidates[0]].astype(str).str.strip()
        ts = pd.to_datetime(combined, errors="coerce")
        if ts.notna().mean() > 0.90:
            return ts

    raise KeyError(f"Could not infer timestamp. Available columns: {list(df.columns)}")


def solar_position_noaa(index_like, lat_deg, lon_deg):
    idx = pd.to_datetime(index_like)
    doy = idx.dayofyear.values.astype(float)
    minutes = (idx.hour.values * 60 + idx.minute.values + idx.second.values / 60.0).astype(float)

    gamma = 2.0 * np.pi / 365.0 * (doy - 1 + (minutes / 60.0 - 12.0) / 24.0)
    decl = (0.006918 - 0.399912 * np.cos(gamma) + 0.070257 * np.sin(gamma)
            - 0.006758 * np.cos(2 * gamma) + 0.000907 * np.sin(2 * gamma)
            - 0.002697 * np.cos(3 * gamma) + 0.00148 * np.sin(3 * gamma))
    eqtime = 229.18 * (0.000075 + 0.001868 * np.cos(gamma) - 0.032077 * np.sin(gamma)
                       - 0.014615 * np.cos(2 * gamma) - 0.040849 * np.sin(2 * gamma))

    tz_offset_hours = lon_deg / 15.0
    tst = (minutes + eqtime + 4.0 * lon_deg - 60.0 * tz_offset_hours) % 1440.0
    hra = np.deg2rad(tst / 4.0 - 180.0)
    lat_r = np.deg2rad(lat_deg)

    cos_zen = np.sin(lat_r) * np.sin(decl) + np.cos(lat_r) * np.cos(decl) * np.cos(hra)
    cos_zen = np.clip(cos_zen, -1.0, 1.0)
    zenith = np.rad2deg(np.arccos(cos_zen))

    az = np.rad2deg(np.arctan2(np.sin(hra), np.cos(hra) * np.sin(lat_r) - np.tan(decl) * np.cos(lat_r))) + 180.0
    az = (az + 360.0) % 360.0

    return pd.DataFrame({"SZ Angle ": zenith, "SA Angle ": az}, index=idx)

In [ ]:
# =========================
# 2. Load source files
# =========================

pv_path = find_file([PV_FILE, "plant_1_generation_data.csv", "Plant_1_Generation.csv"])
weather_path = find_file([WEATHER_FILE, "solar_prediction.csv", "solarprediction.csv"])

pv_raw = pd.read_csv(pv_path)
weather_raw = pd.read_csv(weather_path)

print("PV source:", pv_path, pv_raw.shape)
print("Weather source:", weather_path, weather_raw.shape)
print("PV columns:", list(pv_raw.columns))
print("Weather columns:", list(weather_raw.columns))

In [ ]:
# =========================
# 3. Process weather / solar radiation file
# =========================

weather = weather_raw.copy()
weather["timestamp"] = build_timestamp_from_weather(weather)
weather = weather.dropna(subset=["timestamp"]).copy()

num_cols_weather = weather.select_dtypes(include=[np.number]).columns.tolist()
weather_h = (weather[["timestamp"] + num_cols_weather]
             .set_index("timestamp")
             .resample("1h")
             .mean()
             .reset_index())

weather_h = weather_h.rename(columns={
    "Radiation": "ghi_wm2",
    "Temperature": "temp_f",
    "Humidity": "rh_pct",
    "Speed": "wind_mph",
    "WindDirection(Degrees)": "wind_dir_deg",
})

if "temp_f" in weather_h.columns:
    weather_h["temp_c"] = (weather_h["temp_f"] - 32.0) * 5.0 / 9.0
if "wind_mph" in weather_h.columns:
    weather_h["wind_ms"] = weather_h["wind_mph"] * 0.44704

weather_h["hour"] = weather_h["timestamp"].dt.hour
weather_h["day"] = weather_h["timestamp"].dt.day
weather_h["month"] = weather_h["timestamp"].dt.month

weather_h = weather_h[(weather_h["timestamp"] >= pd.to_datetime(START_TIMESTAMP)) &
                      (weather_h["timestamp"] <= pd.to_datetime(END_TIMESTAMP))].copy()

print("Weather hourly shape:", weather_h.shape)
print(weather_h.head())

In [ ]:
# =========================
# 4. Process PV generation file
# =========================

pv = pv_raw.copy()
if "DATE_TIME" not in pv.columns:
    raise KeyError("PV file must contain DATE_TIME column.")

pv["DATE_TIME"] = pd.to_datetime(pv["DATE_TIME"], format="%d-%m-%Y %H:%M", errors="coerce")
pv = pv.dropna(subset=["DATE_TIME"]).copy()

num_cols_pv = pv.select_dtypes(include=[np.number]).columns.tolist()
pv_h = (pv[["DATE_TIME"] + num_cols_pv]
        .set_index("DATE_TIME")
        .resample("1h")
        .mean()
        .reset_index())

if "AC_POWER" in pv_h.columns:
    pv_h = pv_h.rename(columns={"AC_POWER": "ac_power"})
else:
    candidates = [c for c in pv_h.columns if c.lower().replace(" ", "_") in ["ac_power", "ac_power_kw", "acoutput"]]
    if not candidates:
        raise KeyError(f"Could not find AC power column. Available columns: {list(pv_h.columns)}")
    pv_h = pv_h.rename(columns={candidates[0]: "ac_power"})

pv_h["hour"] = pv_h["DATE_TIME"].dt.hour
pv_day = pv_h[pv_h["ac_power"].notna() & (pv_h["ac_power"] > 0)].copy()

print("PV hourly shape:", pv_h.shape)
print("PV daylight/nonzero samples:", pv_day.shape)
print(pv_h.head())

In [ ]:
# =========================
# 5. Integrate PV output with weather feature table
# =========================

# The source files do not share the same native date range. To build the curated benchmark file,
# this cell assigns representative PV AC output values to weather rows using hour-of-day matching.
# State this transparently in the README/provenance note and manuscript.

rng = np.random.default_rng(RANDOM_SEED)
merged = weather_h.copy()
merged["ac_output"] = np.nan

for h in range(24):
    idx = merged.index[merged["hour"] == h]
    pool = pv_day.loc[pv_day["hour"] == h, "ac_power"].dropna().values
    if len(idx) == 0:
        continue
    if len(pool) == 0:
        merged.loc[idx, "ac_output"] = 0.0
    else:
        merged.loc[idx, "ac_output"] = rng.choice(pool, size=len(idx), replace=True)

print("Merged feature table shape:", merged.shape)
print(merged[["timestamp", "ghi_wm2", "temp_c", "rh_pct", "wind_ms", "wind_dir_deg", "ac_output", "hour", "day", "month"]].head())

In [ ]:
# =========================
# 6. Add solar-position features
# =========================

solar_pos = solar_position_noaa(merged["timestamp"], LATITUDE_DEG, LONGITUDE_DEG).reset_index(drop=True)
merged = merged.reset_index(drop=True)
merged["SZ Angle "] = solar_pos["SZ Angle "]
merged["SA Angle "] = solar_pos["SA Angle "]

print(merged[["timestamp", "SZ Angle ", "SA Angle "]].head())

In [ ]:
# =========================
# 7. Final formatting and export
# =========================

# Cross-platform date formatting
if os.name == "nt":
    timestamp_str = merged["timestamp"].dt.strftime("%#m/%#d/%Y %#H:%M")
else:
    timestamp_str = merged["timestamp"].dt.strftime("%-m/%-d/%Y %-H:%M")

final = pd.DataFrame({
    "Timestamp": timestamp_str,
    "GHI ": merged["ghi_wm2"],
    "Temperature ": merged["temp_c"],
    "Relative Humidity ": merged["rh_pct"],
    "Wind Speed ": merged["wind_ms"],
    "Wind Direction": merged["wind_dir_deg"],
    "Hour": merged["hour"].astype(int),
    "Day": merged["day"].astype(int),
    "Month": merged["month"].astype(int),
    "SZ Angle ": merged["SZ Angle "],
    "SA Angle ": merged["SA Angle "],
    "AC Output": merged["ac_output"],
})

final = final.dropna().reset_index(drop=True)
final.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Final shape:", final.shape)
print("Missing values:", int(final.isna().sum().sum()))
print("Duplicate rows:", int(final.duplicated().sum()))
print(final.head())

In [ ]:
# =========================
# 8. Validation of the published final CSV, if available
# =========================

published_file = "Solar_Weather_Data_Processed.csv"
if os.path.exists(published_file):
    pub = pd.read_csv(published_file)
    pub_ts = pd.to_datetime(pub["Timestamp"], errors="coerce")
    expected_range = pd.date_range(pub_ts.min(), pub_ts.max(), freq="1h")
    missing_hours = sorted(set(expected_range) - set(pub_ts))

    print("Published final shape:", pub.shape)
    print("Published date range:", pub_ts.min(), "to", pub_ts.max())
    print("Published missing values:", int(pub.isna().sum().sum()))
    print("Published duplicate rows:", int(pub.duplicated().sum()))
    print("Published hourly timestamp gaps:", len(missing_hours))
else:
    print("Published final file not found in the current folder. Validation skipped.")

## Manuscript wording to use

Use this wording in the paper's dataset section:

> The dataset used in this study was prepared by integrating publicly available photovoltaic generation and solar-weather datasets obtained from Kaggle. The final processed dataset contains hourly solar-weather features and photovoltaic AC power output values and has been deposited in Mendeley Data together with the preprocessing code and data dictionary.

Do **not** describe `AC Output` as kWh. It is a power-output target derived from the `AC_POWER` column of the PV generation dataset.